In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/tmp/ipykernel_2598295/2463364532.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

ImportError: cannot import name 'GradScaler' from 'torch.amp' (/data/inouey2/conda/lib/python3.10/site-packages/torch/amp/__init__.py)

In [ ]:
data_type = 'drug'

In [ ]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
cell_sum = np.sum(res.T, axis=1)
drug_sum = np.sum(res.T, axis=0)

if data_type == 'drug':
    target_dim = [
        # 0,  # Cell
        1  # Drug
    ]
else:
    target_dim = [
        0,  # Cell
        # 1  # Drug
    ]

In [ ]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [ ]:
method = "GAT"
params = get_params(method, f"NCI_{method}_New")
PATH = f"../{name}_data/"
params.update(
    {
        "n_drug": drug_sim.shape[0],
        "n_cell": cell_sim.shape[0],
        "n_gene": gene_sim.shape[0],
        "epochs": 1,
        "params_heads": 1,
        "params_hidden1": 128,
    }
)

In [ ]:
n_kfold = 1
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        print(dim)
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue
        epochs = []
        true_data_s = pd.DataFrame()
        predict_data_s = pd.DataFrame()
        for fold in range(n_kfold):
            true_data, predict_data = drGAT_new(
                res_mat=res,
                null_mask=null_mask.values,
                target_dim=dim,
                target_index=target_index,
                S_d=drug_sim,
                S_c=cell_sim,
                S_g=gene_sim,
                A_cg=A_cg,
                A_dg=A_dg,
                PATH=PATH,
                params=params,
                device=device,
                seed=seed,
            )

            true_data_s = pd.concat(
                [true_data_s, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_data_s = pd.concat(
                [predict_data_s, pd.DataFrame(predict_data).T], ignore_index=True
            )

In [ ]:
true_datas.to_csv(f"new_true_{data_type}_{method}_{args.data}.csv")
predict_datas.to_csv(f"new_{data_type}_pred_{method}_{args.data}.csv")